In [ ]:
### libraries
library(data.table)
library(tidyverse)
library(ggrepel)
library(patchwork)

In [ ]:
source("../workflow/scripts/init.R")
source("../workflow/scripts/QN_functions.R")

In [ ]:
theme_set(theme_bw())

In [ ]:
cava1_dir <- "<path_to_processed_data>/human_cava_cohort_plasma_1/"
cava2_dir <- "<path_to_processed_data>/human_cava_cohort_plasma_2/"

In [ ]:
outdir <- "<path_to_processed_data>/human_cava_1_cava_2/"
dir.create(outdir)

In [ ]:
colors_significance <- c("cava1" = "#1a9850", "cava2" = "#a6cee3",
                       "both" = "#fb9a99", 
                         "none" = "grey")

In [ ]:
sex <- "Female"

In [ ]:
dir.create(file.path(outdir, paste0("volcano_", sex)))

In [ ]:
df_1 <- read_volcano_df(file.path(cava1_dir, "volcano", paste0("By_Sex_", sex), 
                                   "Gensini", "logFC_df.csv"), "cava1")

In [ ]:
df_2 <- read_volcano_df(file.path(cava2_dir, "volcano", paste0("By_Sex_", sex), 
                                   "Gensini", "logFC_df.csv"), "cava2")

In [ ]:
combined_df <- full_join(df_1, df_2)

In [ ]:
head(combined_df)

In [ ]:
combined_df$sign <- "none"

combined_df[pval_cava1 < 0.05, sign:= "cava1",]

combined_df[pval_cava2 < 0.05, sign:= "cava2",]

combined_df[pval_cava2 < 0.05 & pval_cava1 < 0.05, sign:= "both",]

In [ ]:
ggplot(combined_df, aes(x = log2fc_cava1, y = log2fc_cava2)) + geom_point(aes( color = sign)) + 
    coord_equal() + 
    scale_color_manual(values = colors_significance) + 
     geom_text_repel(data = combined_df[(abs(log2fc_cava1) > 0.5 & abs(log2fc_cava2) > 0.5) ], 
                        aes(x = log2fc_cava1, y = log2fc_cava2, label = sequence_label_short))
 ggsave(file.path(outdir,  paste0("volcano_", sex),  "logFC_compare.pdf"), width = 8, height = 8)

In [ ]:
write.csv(combined_df, file.path(outdir,  paste0("volcano_", sex),  "logFC_combined.csv"))

### merging data and metadata to create a combined file for correlations and differential analysis

In [ ]:
df_cava1 <- fread(file.path(cava1_dir, "normalised_quantiles","full_samples_normalised.csv"))
df_cava1

In [ ]:
df_cava2 <- fread(file.path(cava2_dir, "normalised_quantiles","full_samples_normalised.csv"))
df_cava2

In [ ]:
df_full <- full_join(df_cava1, df_cava2)

In [ ]:
cava1_long <- pivot_longer(df_cava1, !c(coordinate_unique, sequence_label, library, color_col))
cava1_long$cava <- "cava1"

In [ ]:
cava2_long <- pivot_longer(df_cava2, !c(coordinate_unique, sequence_label, library, color_col))
cava2_long$cava <- "cava2"

In [ ]:
cava_long <- rbind(cava1_long, cava2_long)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5)
ggplot(cava_long, aes(x = value, color = cava)) + geom_density(fill = NA) + facet_wrap(~library)

In [ ]:
cava_long %>% group_by(cava, library, coordinate_unique) %>%
    summarize(mean_int = mean(value)) %>% 
    pivot_wider(values_from = mean_int, names_from = cava) %>%
    ggplot() + aes(x = cava1, y = cava2, color = library) + 
        geom_point() + coord_equal() + 
    geom_text_repel(aes(label = coordinate_unique)) + 
    ggtitle("mean_peptide_intensities")

In [ ]:
dir.create(file.path(outdir, "combined_analysis", "normalised_quantiles"), recursive = TRUE)

In [ ]:
my_wt(df_full, file.path(outdir, "combined_analysis", "normalised_quantiles", "full_samples_normalised.csv"))

In [ ]:
annot_df <- as.data.frame(unique(cava_long[,c('name', 'cava')]))

In [ ]:
colnames(annot_df)[1] <- 'sample_name'

In [ ]:
cols_to_use <- annot_df$sample_name

In [ ]:
annot_df$organism <- "human"

In [ ]:
annot_df$tissue <- "plasma"

In [ ]:
write.table(annot_df,
     "../config/human_cava_cohort_combined.csv",
           sep = ",", quote = F, row.names = F)

### merged annotation

In [ ]:
annot_cava_1 <- fread("<path_to_metadata>/Clinical_data_CAVA_COHORT_JD_all_parameter_full.csv")

In [ ]:
annot_cava_2 <- fread("<path_to_metadata>/Justine_clinical_data_CAVA_cohort2_2025.csv")

In [ ]:
### clean up metadata
colnames(annot_cava_2) <- gsub("Event Name", "Event name", colnames(annot_cava_2))

In [ ]:
colnames(annot_cava_2) <- gsub("Maximum Stenosis \\(%\\)", "MaximumStenosis%", colnames(annot_cava_2))

In [ ]:
colnames(annot_cava_2) <- gsub("TAV \\(mm2\\)", "TAVmm2", colnames(annot_cava_2))
colnames(annot_cava_2) <- gsub("Fatty \\(%\\)", "Fatty%", colnames(annot_cava_2))
colnames(annot_cava_2) <- gsub("Atheroma Burden \\(%\\)", "AtheromaBurden%", colnames(annot_cava_2))
colnames(annot_cava_2) <- gsub("Fibrous \\(%\\)", "Fibrous%", colnames(annot_cava_2))
colnames(annot_cava_2) <- gsub("Necrotic \\(%\\)", "Necrotic%", colnames(annot_cava_2))
colnames(annot_cava_2) <- gsub("Calcium \\(%\\)", "Calcium%", colnames(annot_cava_2))
colnames(annot_cava_2) <- gsub("Gensini Score", "GensiniScore", colnames(annot_cava_2))
colnames(annot_cava_2) <- gsub("Fibrous \\(%\\)", "Fibrous%", colnames(annot_cava_2))
colnames(annot_cava_2) <- gsub("Statin", "Statin_use", colnames(annot_cava_2))
colnames(annot_cava_2) <- gsub("Lymphocytes  Absolute Count \\(k\\/uL\\)", "Lymphocytes Absolute Count \\(k\\/uL\\)", colnames(annot_cava_2))
colnames(annot_cava_2) <- gsub("Total Cholesterol \\(mg/dL\\)", "Totalcholesterolmgdl", colnames(annot_cava_2))

In [ ]:
annot_cava_1$cava <- "cava1"
annot_cava_2$cava <- "cava2"

In [ ]:
annot_cava <- rbind(annot_cava_1, annot_cava_2, fill = TRUE)

In [ ]:
write.table(annot_cava,
          sep = ",", 
          quote = FALSE, 
          row.names = FALSE,
          file.path("<path_to_metadata>/peptide_array/metadata/CAVA_combined.csv"))

In [ ]:
annot_cava_2$Sample_full_ID <- gsub('15328_186', '15328_186_cava2',annot_cava_2$Sample_full_ID )

In [ ]:
annot_df <- left_join(annot_df, 
          rbind(annot_cava_1[, c("Sample_full_ID", "Sex")], 
                annot_cava_2[, c("Sample_full_ID", "Sex")], fill = TRUE),
          by = c("sample_name" = "Sample_full_ID"))

In [ ]:
### making a PCA 
df_rot <- run_pca(df_full[,..cols_to_use], 
                  file.path(outdir, "combined_analysis", "normalised_quantiles"))
df_rot$sample_name <- as.character(row.names(df_rot))
annot_df$sample_name <- as.character(annot_df$sample_name)


In [ ]:
df_rot <- left_join(df_rot, annot_df)

In [ ]:
ggplot(df_rot, aes(x = PC1, y = PC2, color = cava)) + geom_point() + 
    geom_text_repel(aes(label = sample_name), size = 2) 
ggsave(file.path(outdir, "combined_analysis", "normalised_quantiles",  "PCA_plot.pdf"), width = 5, height = 5)

In [ ]:
desired_cols <- "Sex"

In [ ]:
unwanted_categorical_cols <- c("cava")

In [ ]:
annot <- data.frame(annot_df[,c(2,5)])

In [ ]:
row.names(annot) <- annot_df$sample_name

In [ ]:
design <- model.matrix(reformulate(desired_cols), data = annot)

In [ ]:
batch <- annot[[unwanted_categorical_cols[1]]]

In [ ]:
batch2 <- NULL

In [ ]:
covariates <- NULL

In [ ]:
library(limma)

In [ ]:
# integrate data / remove batch effect
integrated_data <- removeBatchEffect(as.matrix(df_full[,..cols_to_use]), 
                                     batch = batch, 
                                     batch2 = batch2, 
                                     covariates = covariates, 
                                     design = design
                                    )

In [ ]:
dir.create(file.path(outdir, "combined_analysis", "normalised_quantiles_corrected"))

In [ ]:
df_rot <- run_pca(integrated_data, 
                  file.path(outdir, "combined_analysis", "normalised_quantiles_corrected"))

In [ ]:
df_rot$sample_name <- as.character(row.names(df_rot))
df_rot <- left_join(df_rot, annot_df)

In [ ]:
ggplot(df_rot, aes(x = PC1, y = PC2, color = cava)) + geom_point() + 
    geom_text_repel(aes(label = sample_name), size = 2) 
ggsave(file.path(outdir, "combined_analysis", 
                 "normalised_quantiles_corrected",  "PCA_plot.pdf"), width = 5, height = 5)

In [ ]:
ggplot(df_rot, aes(x = PC1, y = PC2, color = Sex)) + geom_point() + 
    geom_text_repel(aes(label = sample_name), size = 2) 
ggsave(file.path(outdir, "combined_analysis", "normalised_quantiles_corrected",  "PCA_plot_by_Sex.pdf"), width = 5, height = 5)

In [ ]:
setdiff(colnames(df_full), colnames(integrated_data))

In [ ]:
integrated_data <- as.data.frame(integrated_data)

In [ ]:
integrated_data$coordinate_unique <- df_full$coordinate_unique
integrated_data$sequence_label <- df_full$sequence_label
integrated_data$library <- df_full$library
integrated_data$color_col <- df_full$color_col

In [ ]:
my_wt(integrated_data, 
     file.path(outdir, "combined_analysis", "normalised_quantiles", "integrated", "full_samples_normalised.csv"))